In [ ]:
import time
import csv
import pandas as pd
import os
import sys
import random
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys # Cần import cái này
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ================= CẤU HÌNH =================
INPUT_CSV = "input_urls.csv"
OUTPUT_CSV = "fb_2.csv"
PROFILE_PATH = r"C:\selenium_fb_clean_v2" 
MAX_CYCLES = 500 
# ============================================

def setup_driver():
    print("🛠 Đang khởi động Chrome Desktop...")
    chrome_options = Options()
    chrome_options.add_argument(f"user-data-dir={PROFILE_PATH}")
    
    prefs = {"profile.default_content_setting_values.notifications": 2}
    chrome_options.add_experimental_option("prefs", prefs)
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_argument("--start-maximized")
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
        return driver
    except Exception as e:
        print(f"❌ Lỗi mở Chrome: {e}")
        sys.exit()

def click_js(driver, element):
    driver.execute_script("arguments[0].click();", element)

def switch_to_all_comments(driver):
    """Chuyển bộ lọc sang Tất cả bình luận"""
    print("⚙️ Đang thử chuyển bộ lọc...")
    try:
        filter_menu = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'Phù hợp nhất') or contains(text(), 'Most relevant')]"))
        )
        click_js(driver, filter_menu)
        time.sleep(2)
        
        all_comments_opt = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'Tất cả bình luận') or contains(text(), 'All comments')]"))
        )
        click_js(driver, all_comments_opt)
        print("   ✅ Đã chọn: Tất cả bình luận")
        time.sleep(3)
    except Exception:
        print("   ℹ️ Không cần chuyển bộ lọc (hoặc không tìm thấy).")

def expand_everything(driver):
    """Click mở rộng Reply và Xem thêm"""
    try:
        # Nút Xem thêm bình luận (Pagination)
        view_more_btns = driver.find_elements(By.XPATH, "//span[contains(text(), 'Xem thêm bình luận') or contains(text(), 'Xem các bình luận trước') or contains(text(), 'View more comments')]")
        # Nút Phản hồi (Reply)
        view_replies_btns = driver.find_elements(By.XPATH, "//div[@role='button']//span[contains(text(), 'phản hồi') or contains(text(), 'replies')]")
        # Nút Xem thêm text dài
        see_more_text_btns = driver.find_elements(By.XPATH, "//div[@role='button'][contains(text(), 'Xem thêm') or contains(text(), 'See more')]")

        all_buttons = view_more_btns + view_replies_btns + see_more_text_btns
        
        count = 0
        for btn in all_buttons:
            try:
                if btn.is_displayed():
                    click_js(driver, btn)
                    count += 1
                    time.sleep(0.2)
            except:
                pass
        return count
    except:
        return 0

def smart_scroll(driver, retry_count):
    """
    Kỹ thuật cuộn đàn hồi:
    1. Nhấn phím END để xuống đáy.
    2. Nếu không thấy thay đổi, cuộn lên 1 chút (PAGE_UP) rồi cuộn xuống lại.
    """
    body = driver.find_element(By.TAG_NAME, "body")
    
    # Cách 1: Nhấn END (Mạnh hơn scrollBy)
    body.send_keys(Keys.END)
    time.sleep(2) # Chờ load cơ bản
    
    # Nếu đang kẹt (retry_count cao), dùng tuyệt chiêu "Rung lắc"
    if retry_count >= 2:
        # Cuộn lên khoảng 30% màn hình
        driver.execute_script("window.scrollBy(0, -400);")
        time.sleep(1)
        # Cuộn xuống lại
        body.send_keys(Keys.END)
        time.sleep(3) # Chờ lâu hơn chút

def process_post(driver, url):
    print(f"\n🌐 Đang xử lý: {url}")
    driver.get(url)
    time.sleep(3)
    
    switch_to_all_comments(driver)
    
    collected_comments = set()
    
    last_height = driver.execute_script("return document.body.scrollHeight")
    stuck_counter = 0      # Đếm số lần bị kẹt
    
    print("🚀 Bắt đầu quét... (Sử dụng kỹ thuật cuộn đàn hồi)")

    for cycle in range(MAX_CYCLES):
        # 1. Mở rộng (Reply/More)
        expand_count = expand_everything(driver)
        
        # 2. Cào dữ liệu
        current_batch = 0
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, "div[dir='auto']")
            for el in elements:
                try:
                    text = el.text.strip()
                    if len(text) > 0 and text not in ["Viết bình luận...", "Đang tải...", "Phù hợp nhất"]:
                        if text not in collected_comments:
                            collected_comments.add(text)
                            current_batch += 1
                except:
                    continue
        except:
            pass
        
        # In trạng thái
        sys.stdout.write(f"\r   🔄 Vòng {cycle} | Tổng: {len(collected_comments)} cmt | Mới: {current_batch} | Kẹt: {stuck_counter}/7   ")
        sys.stdout.flush()
        
        # 3. THỰC HIỆN CUỘN
        smart_scroll(driver, stuck_counter)
        
        # 4. KIỂM TRA ĐỘ CAO MỚI
        new_height = driver.execute_script("return document.body.scrollHeight")
        
        if new_height == last_height and current_batch == 0:
            stuck_counter += 1
        else:
            stuck_counter = 0 # Reset nếu thấy chiều cao tăng HOẶC có comment mới
            last_height = new_height

        # ĐIỀU KIỆN DỪNG: Tăng lên 7 lần thử liên tiếp (kiên nhẫn hơn)
        if stuck_counter >= 7:
            # Check kỹ lần cuối: Tìm xem có spinner "Đang tải" không
            try:
                loaders = driver.find_elements(By.XPATH, "//*[contains(text(), 'Đang tải') or contains(@class, 'spinner')]")
                if len(loaders) > 0:
                    stuck_counter = 4 # Reset nhẹ, cho thêm cơ hội vì đang tải
                    time.sleep(3)
                    continue
            except:
                pass

            print(f"\n   🛑 Đã dừng vì không tải thêm được nội dung mới sau 7 lần thử.")
            break

    return list(collected_comments)

def main():
    if not os.path.exists(INPUT_CSV):
        print("❌ Thiếu input_urls.csv")
        return

    os.system("taskkill /F /IM chrome.exe /T >nul 2>&1")
    driver = setup_driver()
    
    try:
        # Check login
        driver.get("https://www.facebook.com")
        time.sleep(2)
        if "login" in driver.current_url:
             print("\n❌ CẢNH BÁO: CHƯA ĐĂNG NHẬP!")
             input("👉 Nhấn ENTER sau khi đăng nhập xong...")

        try:
            df = pd.read_csv(INPUT_CSV)
            urls = df['url'].tolist()
        except:
            return

        for url in urls:
            clean_url = url.replace("mbasic.facebook.com", "www.facebook.com").replace("m.facebook.com", "www.facebook.com")
            
            comments = process_post(driver, clean_url)
            
            if comments:
                df_out = pd.DataFrame({'url': [clean_url]*len(comments), 'cmt': comments})
                mode = 'a' if os.path.exists(OUTPUT_CSV) else 'w'
                header = False if os.path.exists(OUTPUT_CSV) else True
                df_out.to_csv(OUTPUT_CSV, mode=mode, header=header, index=False, encoding='utf-8-sig')
                print(f"   💾 Đã lưu {len(comments)} dòng.")
            else:
                print("   ⚠️ Không lấy được data.")

    except Exception as e:
        print(f"\n❌ ERROR: {e}")
    finally:
        print("\nHoàn tất.")

if __name__ == "__main__":
    main()

🛠 Đang khởi động Chrome Desktop...

🌐 Đang xử lý: https://www.facebook.com/BBCnewsVietnamese/posts/pfbid02YMNR1LDpxshp1iwN6ajW47GKizYKdghJoMKhjzg16UQpaMW9fjhx6KoCyVgJCwHSl?locale=vi_VN
⚙️ Đang thử chuyển bộ lọc...
   ✅ Đã chọn: Tất cả bình luận
🚀 Bắt đầu quét... (Sử dụng kỹ thuật cuộn đàn hồi)
   🔄 Vòng 35 | Tổng: 1374 cmt | Mới: 0 | Kẹt: 6/7     
   🛑 Đã dừng vì không tải thêm được nội dung mới sau 7 lần thử.
   💾 Đã lưu 1374 dòng.

Hoàn tất.


In [2]:
import csv
import logging
import os
import sys
import time
from typing import List, Set

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# ================= CẤU HÌNH (CONFIGURATION) =================
INPUT_CSV = "input_urls.csv"
OUTPUT_CSV = "fb.csv"
PROFILE_PATH = r"C:\selenium_fb_clean_v2"
MAX_SCROLL_ATTEMPTS = 50
SCROLL_PAUSE_TIME = 2.0
WAIT_TIMEOUT = 10

# Cấu hình Logging (Không icon, hiển thị thời gian)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

class FacebookScraper:
    def __init__(self, profile_path: str):
        self.profile_path = profile_path
        self.driver = self._setup_driver()

    def _setup_driver(self) -> webdriver.Chrome:
        """Khởi tạo Chrome Driver với profile và options tối ưu."""
        logger.info("Dang khoi tao Chrome Driver...")
        options = Options()
        options.add_argument(f"user-data-dir={self.profile_path}")
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("prefs", {
            "profile.default_content_setting_values.notifications": 2  # Block notifications
        })
        options.add_argument("--start-maximized")

        try:
            service = Service(ChromeDriverManager().install())
            return webdriver.Chrome(service=service, options=options)
        except Exception as e:
            logger.error(f"Khong the khoi dong Chrome: {e}")
            sys.exit(1)

    def _click_element(self, element) -> None:
        """Wrapper an toàn cho việc click bằng Javascript."""
        try:
            self.driver.execute_script("arguments[0].click();", element)
        except Exception:
            pass

    def switch_to_all_comments(self) -> None:
        """Chuyển bộ lọc comment sang 'Tất cả bình luận'."""
        logger.info("Dang chuyen bo loc sang 'Tat ca binh luan'...")
        try:
            # Tìm menu bộ lọc (Phù hợp nhất / Most relevant)
            filter_menu = WebDriverWait(self.driver, 5).until(
                EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'Phù hợp nhất') or contains(text(), 'Most relevant')]"))
            )
            self._click_element(filter_menu)
            time.sleep(1)

            # Chọn 'Tất cả bình luận'
            all_comments_opt = WebDriverWait(self.driver, 5).until(
                EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'Tất cả bình luận') or contains(text(), 'All comments')]"))
            )
            self._click_element(all_comments_opt)
            time.sleep(3) # Chờ reload ajax
            logger.info("Da chuyen bo loc thanh cong.")
        except Exception:
            logger.warning("Khong tim thay bo loc hoac da o che do All Comments.")

    def expand_replies_and_more(self) -> int:
        """Tìm và click các nút 'Xem thêm', 'Phản hồi'."""
        # XPaths cho các nút mở rộng
        xpaths = [
            "//span[contains(text(), 'Xem thêm bình luận') or contains(text(), 'View more comments')]",
            "//div[@role='button']//span[contains(text(), 'phản hồi') or contains(text(), 'replies')]",
            "//div[@role='button'][contains(text(), 'Xem thêm') or contains(text(), 'See more')]"
        ]
        
        clicked_count = 0
        for xpath in xpaths:
            elements = self.driver.find_elements(By.XPATH, xpath)
            for btn in elements:
                if btn.is_displayed():
                    self._click_element(btn)
                    clicked_count += 1
                    time.sleep(0.1) # Delay nhỏ để tránh lag UI
        return clicked_count

    def extract_comments(self) -> Set[str]:
        """Quét và lấy text từ các thẻ div chứa nội dung."""
        comments = set()
        elements = self.driver.find_elements(By.CSS_SELECTOR, "div[dir='auto']")
        
        ignored_phrases = {"Viết bình luận...", "Đang tải...", "Phù hợp nhất", "Tất cả bình luận"}

        for el in elements:
            try:
                text = el.text.strip()
                if text and text not in ignored_phrases:
                    comments.add(text)
            except Exception:
                continue
        return comments

    def process_post(self, url: str) -> List[str]:
        """Logic chính để xử lý một bài viết."""
        logger.info(f"Dang xu ly URL: {url}")
        self.driver.get(url)
        time.sleep(3)

        self.switch_to_all_comments()

        collected_data = set()
        last_height = self.driver.execute_script("return document.body.scrollHeight")
        stuck_count = 0

        for cycle in range(MAX_SCROLL_ATTEMPTS):
            # 1. Mở rộng nội dung
            self.expand_replies_and_more()

            # 2. Thu thập dữ liệu hiện tại
            current_comments = self.extract_comments()
            new_items = len(current_comments) - len(collected_data)
            collected_data.update(current_comments)

            logger.info(f"Vong {cycle+1}/{MAX_SCROLL_ATTEMPTS} | Tong: {len(collected_data)} | Moi: {new_items}")

            # 3. Cuộn trang
            self.driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
            time.sleep(SCROLL_PAUSE_TIME)

            # 4. Kiểm tra chiều cao trang (Logic thoát vòng lặp)
            new_height = self.driver.execute_script("return document.body.scrollHeight")
            
            if new_height == last_height:
                stuck_count += 1
                # Thử cuộn lên một chút rồi cuộn xuống (Wiggle scroll)
                self.driver.execute_script("window.scrollBy(0, -300);")
                time.sleep(1)
                self.driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
                
                if stuck_count >= 5: # Dừng nếu kẹt quá 5 lần
                    logger.info("Khong tai them duoc noi dung. Dung quet.")
                    break
            else:
                stuck_count = 0
                last_height = new_height

        return list(collected_data)

    def close(self):
        """Đóng trình duyệt."""
        if self.driver:
            self.driver.quit()
            logger.info("Da dong trinh duyet.")

def save_to_csv(data: List[dict], filepath: str):
    """Lưu dữ liệu vào CSV (Append mode)."""
    file_exists = os.path.exists(filepath)
    mode = 'a' if file_exists else 'w'
    
    with open(filepath, mode, newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=['url', 'comment'])
        if not file_exists:
            writer.writeheader()
        writer.writerows(data)

def main():
    # 1. Kiểm tra input
    if not os.path.exists(INPUT_CSV):
        logger.error(f"File {INPUT_CSV} khong ton tai.")
        return

    # 2. Kill Chrome cũ (Windows)
    os.system("taskkill /F /IM chrome.exe /T >nul 2>&1")

    # 3. Khởi tạo Scraper
    scraper = FacebookScraper(PROFILE_PATH)

    try:
        # Kiểm tra login thủ công
        scraper.driver.get("https://www.facebook.com")
        if "login" in scraper.driver.current_url:
            logger.warning("Vui long dang nhap Facebook thu cong, sau do nhat Enter...")
            input()

        # Đọc danh sách URL
        df = pd.read_csv(INPUT_CSV)
        urls = df['url'].tolist()

        for url in urls:
            clean_url = url.replace("mbasic.facebook.com", "www.facebook.com").replace("m.facebook.com", "www.facebook.com")
            
            comments = scraper.process_post(clean_url)

            if comments:
                # Chuẩn bị dữ liệu để lưu
                rows = [{'url': clean_url, 'comment': cmt} for cmt in comments]
                save_to_csv(rows, OUTPUT_CSV)
                logger.info(f"Da luu {len(rows)} comment vao CSV.")
            else:
                logger.warning("Khong thu thap duoc comment nao.")

    except Exception as e:
        logger.error(f"Loi chuong trinh: {e}")
    finally:
        scraper.close()

if __name__ == "__main__":
    main()

09:34:05 [INFO] Dang khoi tao Chrome Driver...
09:34:05 [INFO] ====== WebDriver manager ======
09:34:06 [INFO] Get LATEST chromedriver version for google-chrome
09:34:06 [INFO] Get LATEST chromedriver version for google-chrome
09:34:06 [INFO] Driver [C:\Users\Quyen\.wdm\drivers\chromedriver\win64\144.0.7559.109\chromedriver-win32/chromedriver.exe] found in cache
09:34:10 [INFO] Dang xu ly URL: https://www.facebook.com/share/p/14X1oefiCWb/
09:34:18 [INFO] Dang chuyen bo loc sang 'Tat ca binh luan'...
09:34:22 [INFO] Da chuyen bo loc thanh cong.
09:34:25 [INFO] Vong 1/50 | Tong: 77 | Moi: 77
09:34:31 [INFO] Vong 2/50 | Tong: 106 | Moi: 29
09:34:38 [INFO] Vong 3/50 | Tong: 109 | Moi: 3
09:34:43 [INFO] Vong 4/50 | Tong: 151 | Moi: 42
09:34:48 [INFO] Vong 5/50 | Tong: 152 | Moi: 1
09:34:51 [INFO] Khong tai them duoc noi dung. Dung quet.
09:34:51 [INFO] Da luu 152 comment vao CSV.
09:34:51 [INFO] Dang xu ly URL: https://www.facebook.com/share/p/1AaXYSW7Cs/
09:34:57 [INFO] Dang chuyen bo loc 

In [3]:
import pandas as pd
df = pd.read_csv("fb.csv")
df.head()

,url,comment
0,https://www.facebook.com/share/p/17n4BRucaH/,Ren Zhi\n:)))) mấy con gà bt gì về ếch và báo
1,https://www.facebook.com/share/p/17n4BRucaH/,https://www.youtube.com/watch?v=pw8OMQPbCuw
2,https://www.facebook.com/share/p/17n4BRucaH/,Nhiều thằng ko hiểu vào ẳng như nước sôi đổ vô...
3,https://www.facebook.com/share/p/17n4BRucaH/,Bảo Bảo\ntus nó nói về cái gì
4,https://www.facebook.com/share/p/17n4BRucaH/,Warren Mike Nguyen\nanh nào chạy air blade nha...
